# SciCite: End-to-End Pipeline & Training
## Data Pipeline, Live EDA Plotting, Baseline Tuning, Transformer Fine-Tuning & Model Publishing

**Dataset:** `allenai/scicite` (Official splits from NAACL 2019 / Hugging Face)  
**Target Classes:** `method`, `background`, `result`  

This notebook provides the complete, self-contained execution pipeline for the project. It runs the entire machine learning workflow from raw data ingestion and live dynamic plotting to model fine-tuning, local artifact persistence, and public Hugging Face Hub publishing.

In [ ]:
# 1. Environment Setup, Reproducibility & Hardware Check
from pathlib import Path
import pandas as pd
import torch

from src.data import get_data_splits, ID2LABEL
from src.utils import (
    set_seed,
    plot_class_distribution,
    plot_canonical_section_contingency,
    plot_confusion_matrices,
)
from src.models import (
    train_baseline,
    train_transformer,
    save_transformer_model,
    save_baseline_pipeline,
    push_model_to_hub,
)

set_seed(42)
PROJECT_ROOT = Path(".").resolve()

# Hardware & Precision Check
print("=" * 70)
print("ENVIRONMENT & HARDWARE VERIFICATION")
print("=" * 70)
print(f"PyTorch Version:      {torch.__version__}")
print(f"CUDA Available:       {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device:           {torch.cuda.get_device_name(0)}")
    print(f"VRAM Capacity:        {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"BF16 Precision:       {torch.cuda.is_bf16_supported()}")

# Read Hugging Face token from .env for model publishing
HF_TOKEN = None
env_path = PROJECT_ROOT / ".env"
if env_path.is_file():
    for line in env_path.read_text().splitlines():
        if line.startswith("HF_TOKEN="):
            HF_TOKEN = line.split("=", 1)[1].strip()
if HF_TOKEN:
    print("Hugging Face Token:   Configured (Publishing enabled)")
else:
    print("Hugging Face Token:   Not found (Local saving only)")
print("=" * 70)


---
## 1. Data Ingestion & Section Normalization

The dataset is loaded from the official NAACL 2019 publication partitions (`train`, `dev`, `test`).
Raw academic section headers (`sectionName`) are cleaned and mapped into 7 canonical categories (`introduction`, `method`, `result`, `discussion`, `result_discussion`, `other`, `missing`). Two comparative representations are built:
- **`text_only`**: Citation sentence string alone.
- **`text_section_aware`**: Structured conditioned prefix: `"Section: {canonical_section}. Citation: {text}"`.

In [ ]:
# Load and build preprocessed dataset splits
train_df, val_df, test_df = get_data_splits(PROJECT_ROOT)

print("Official SciCite dataset splits loaded successfully:")
print(f"  - Train partition:       {len(train_df):>5} samples")
print(f"  - Validation partition:  {len(val_df):>5} samples")
print(f"  - Test partition:        {len(test_df):>5} samples")

# Verify sample record
sample = train_df.iloc[0]
print("\nSample Data Representation:")
print(f"  Label:            {sample['label_name'].upper()}")
print(f"  Raw Section:      '{sample['sectionName']}' -> Canonical: '{sample['section_clean']}'")
print(f"  Text-only:        {sample['text_only'][:90]}...")
print(f"  Section-aware:    {sample['text_section_aware'][:110]}...")


---
## 2. Live Exploratory Data Analysis & Dynamic Plot Generation

Generate and save the publication-quality figures directly from the data splits rather than reading pre-rendered images.

In [ ]:
# Generate and save publication figures directly
fig_dir = PROJECT_ROOT / "outputs" / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)

dfs = {"train": train_df, "validation": val_df, "test": test_df}

print("[1/2] Plotting class distribution across splits...")
fig_dist = plot_class_distribution(dfs, id2label=ID2LABEL, output_path=fig_dir / "class_distribution.png")
display(fig_dist)

print("[2/2] Plotting conditional intent distribution P(Intent | Section)...")
fig_cont = plot_canonical_section_contingency(train_df, output_path=fig_dir / "canonical_section_contingency.png")
display(fig_cont)


---
## 3. Baseline Model: TF-IDF + Logistic Regression

A classical baseline using sublinear TF-IDF (1–2 n-grams, max 10,000 features, `min_df=2`) and Logistic Regression with validation tuning over regularization strength $C \in [0.5, 1.0, 2.0, 5.0]$ and class weighting.

In [ ]:
# Train baseline models (Text-only and Section-aware)
print("Training TF-IDF + Logistic Regression baselines...")
pipe_text, pred_text, m_text = train_baseline(train_df, val_df, test_df, text_column="text_only", seed=42)
pipe_sec, pred_sec, m_sec = train_baseline(train_df, val_df, test_df, text_column="text_section_aware", seed=42)

# Save fitted section-aware baseline model to disk
models_dir = PROJECT_ROOT / "outputs" / "models"
models_dir.mkdir(parents=True, exist_ok=True)
baseline_save_path = save_baseline_pipeline(pipe_sec, models_dir / "baseline_tfidf.pkl")
print(f"\nSaved baseline pipeline locally to: {baseline_save_path}")

# Baseline summary
baseline_df = pd.DataFrame([
    {"Mode": "Text-only", "Accuracy": m_text["accuracy"], "Macro F1": m_text["macro_f1"], "Method F1": m_text["per_class"]["method"]["f1"], "Background F1": m_text["per_class"]["background"]["f1"], "Result F1": m_text["per_class"]["result"]["f1"]},
    {"Mode": "Section-aware", "Accuracy": m_sec["accuracy"], "Macro F1": m_sec["macro_f1"], "Method F1": m_sec["per_class"]["method"]["f1"], "Background F1": m_sec["per_class"]["background"]["f1"], "Result F1": m_sec["per_class"]["result"]["f1"]},
])
print("\n=== Baseline Performance Summary ===")
display(baseline_df)


---
## 4. Transformer Fine-Tuning & Model Publishing

Three architectures got evaluated:
1. **ModernBERT** (`answerdotai/ModernBERT-base`, 149M parameters) — modern RoPE-based general transformer.
2. **SciBERT** (`allenai/scibert_scivocab_uncased`, 110M parameters) — scientific domain-specific vocabulary.
3. **DistilBERT** (`distilbert-base-uncased`, 66M parameters) — fast distilled model.

Each fine-tuned model is saved locally to `outputs/models/<name>/` and automatically published as a public model to Hugging Face Hub under `<username>/<name>` using `HF_TOKEN`.

In [ ]:
# Complete Ablation Matrix: All 6 Transformer Training Configurations
TRAINING_CONFIGS = [
    {
        "name": "scicite-modernbert-section-aware",
        "base_model": "answerdotai/ModernBERT-base",
        "text_column": "text_section_aware",
        "description": "ModernBERT Leaderboard Champion (Section-Aware)",
    },
    {
        "name": "scicite-modernbert-text-only",
        "base_model": "answerdotai/ModernBERT-base",
        "text_column": "text_only",
        "description": "ModernBERT General Baseline (Text-Only)",
    },
    {
        "name": "scicite-scibert-text-only",
        "base_model": "allenai/scibert_scivocab_uncased",
        "text_column": "text_only",
        "description": "SciBERT Scientific Domain Model (Best on Result Class)",
    },
    {
        "name": "scicite-scibert-section-aware",
        "base_model": "allenai/scibert_scivocab_uncased",
        "text_column": "text_section_aware",
        "description": "SciBERT Scientific Domain Model (Section-Aware)",
    },
    {
        "name": "scicite-distilbert-section-aware",
        "base_model": "distilbert-base-uncased",
        "text_column": "text_section_aware",
        "description": "DistilBERT Fast Distilled Model (Section-Aware)",
    },
    {
        "name": "scicite-distilbert-text-only",
        "base_model": "distilbert-base-uncased",
        "text_column": "text_only",
        "description": "DistilBERT Lightweight Baseline (Text-Only)",
    },
]

transformer_results = {}
HF_USERNAME = "Tancsi"

for cfg in TRAINING_CONFIGS:
    name = cfg["name"]
    base = cfg["base_model"]
    col = cfg["text_column"]
    desc = cfg["description"]
    
    print(f"\n{'='*75}")
    print(f"Fine-tuning: {name} ({desc})")
    print(f"Base: {base} | Input Mode: {col}")
    print(f"{'='*75}")
    
    exp_dir = PROJECT_ROOT / "experiments" / name
    trainer, preds, metrics = train_transformer(
        model_name=base,
        train_df=train_df,
        val_df=val_df,
        test_df=test_df,
        text_column=col,
        output_dir=exp_dir,
        num_epochs=4,
        batch_size=32,
        learning_rate=2e-5,
    )
    transformer_results[name] = {"metrics": metrics, "predictions": preds}
    print(f"-> Test Accuracy: {metrics['accuracy']:.4f} | Test Macro-F1: {metrics['macro_f1']:.4f}")
    
    # 1. Save locally
    save_dir = PROJECT_ROOT / "outputs" / "models" / name
    save_transformer_model(trainer, trainer.tokenizer, save_dir)
    print(f"-> Saved locally to: {save_dir}")
    
    # 2. Push to Hugging Face Hub (Public)
    if HF_TOKEN:
        repo_id = f"{HF_USERNAME}/{name}"
        print(f"-> Pushing to Hugging Face Hub as public: {repo_id}...")
        url = push_model_to_hub(save_dir, repo_id, token=HF_TOKEN, private=False)
        print(f"-> Published to Hugging Face: {url}")


---
## 5. Master Comparative Leaderboard & Output Export

The master benchmark table compares all fine-tuned configurations on the official unseen test set (N=1,859).

In [ ]:
# Load and display master comparative leaderboard
leaderboard_path = PROJECT_ROOT / "outputs" / "all_models_comparative_summary.csv"
if leaderboard_path.is_file():
    leaderboard = pd.read_csv(leaderboard_path)
    display_cols = ["Model", "Representation", "Accuracy", "Macro F1", "Method F1", "Background F1", "Result F1"]
    display_df = leaderboard[display_cols].sort_values("Macro F1", ascending=False).reset_index(drop=True)
    print("=== Master Comparative Leaderboard (Official Test Split, N=1,859) ===")
    display(display_df)

# Plot composite confusion matrix
test_pred_path = PROJECT_ROOT / "outputs" / "test_predictions.csv"
if test_pred_path.is_file():
    pred_df = pd.read_csv(test_pred_path)
    cm_models = [
        ("Baseline (Text-only)", "pred_text_only"),
        ("Baseline (Section-aware)", "pred_text_section_aware"),
        ("ModernBERT (Section-aware)", "pred_modernbert_text_section_aware"),
        ("SciBERT (Section-aware)", "pred_scibert_text_section_aware"),
    ]
    cm_out = PROJECT_ROOT / "outputs" / "figures" / "confusion_matrices_comparison.png"
    cm_fig = plot_confusion_matrices(pred_df, cm_models, output_path=str(cm_out))
    display(cm_fig)


---
## 6. Pipeline Execution Summary

1. **Full Reproducibility**: Dataset ingested, section headers normalized, and live figures saved to `outputs/figures/`.
2. **Artifacts Persisted Locally**: All 6 fine-tuned Transformer models and the fitted baseline pipeline saved to `outputs/models/`.
3. **Public Deployment**: Models published to [Hugging Face (`Tancsi`)](https://huggingface.co/Tancsi) for zero-credential public evaluation.
4. **Next Step**: Open `03_interactive_inference.ipynb` to test custom sentences interactively with zero training time.